In [12]:
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import joblib
import numpy as np

In [13]:
X_user_train = pd.read_csv("../data/X_user_train_scaled.csv")
X_movie_train = pd.read_csv("../data/X_movie_train_scaled.csv")
y_train = pd.read_csv("../data/y_train_scaled.csv")

X_user_test = pd.read_csv("../data/X_user_test_scaled.csv")
X_movie_test = pd.read_csv("../data/X_movie_test_scaled.csv")
y_test = pd.read_csv("../data/y_test_scaled.csv")

scaler_users = joblib.load('../models/scaler_users.pkl')
scaler_movies = joblib.load('../models/scaler_movies.pkl')
scaler_y = joblib.load('../models/scaler_y.pkl')

In [14]:
print("X_user_train shape:", X_user_train.shape)
print("X_movie_train shape:", X_movie_train.shape)
print("y_train shape:", y_train.shape)

X_user_train shape: (210868, 227)
X_movie_train shape: (210868, 599)
y_train shape: (210868, 1)


In [15]:
#Build initial NN

from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
#poorer results when reg applied, so not used currently

num_outputs = 32

tf.random.set_seed(42)

userNN = tf.keras.models.Sequential([
    Dense(256, activation='relu'),
    Dropout(0.4),

    Dense(128, activation='relu'),
    Dropout(0.4),

    Dense(num_outputs, activation='linear')
])

movieNN = tf.keras.models.Sequential([

    Dense(256, activation='relu'),
    Dropout(0.4),

    Dense(128, activation='relu'),
    Dropout(0.4),

    Dense(num_outputs, activation='linear')
])

#create user input and point to base network
input_user = tf.keras.layers.Input(shape=(X_user_train.shape[1],))
vu = userNN(input_user)
#normalise to unit vector
vu = tf.keras.ops.normalize(vu, axis=1)


#create movie input and point to base network
input_movie = tf.keras.layers.Input(shape=(X_movie_train.shape[1],))
vm = movieNN(input_movie)
#normalise to unit vector
vm = tf.keras.ops.normalize(vm, axis=1)

#dot product layer
output = tf.keras.layers.Dot(axes=1)([vu, vm])

#model spec
model = tf.keras.Model(inputs=[input_user, input_movie], outputs=output)

model.summary()



Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 227)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_6       │ (None, 599)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_2        │ (None, 32)        │     95,392 │ input_layer_4[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_3        │ (None, 32)        │    190,624 │ input_layer_6[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_2         │ (None, 32)        │          0 │ sequential_2[0][… │
│ (Normalize)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_3         │ (None, 32)        │          0 │ sequential_3[0][… │
│ (Normalize)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_1 (Dot)         │ (None, 1)         │          0 │ normalize_2[0][0… │
│                     │                   │            │ normalize_3[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 286,016 (1.09 MB)

 Trainable params: 286,016 (1.09 MB)

 Non-trainable params: 0 (0.00 B)

output = 32

userNN = 
256 -> 128

movieNN = 
256 -> 128

In [16]:
tf.random.set_seed(42)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,loss=cost_fn)

In [17]:
tf.random.set_seed(42)
model.fit([X_user_train, X_movie_train], y_train, epochs=30)

Epoch 1/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - loss: 0.1460
Epoch 2/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.1377
Epoch 3/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.1343
Epoch 4/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.1325
Epoch 5/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.1313
Epoch 6/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.1303
Epoch 7/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.1295
Epoch 8/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - loss: 0.1286
Epoch 9/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.1284
Epoch 10/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.1275
Epoch 11/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - loss: 0.1269
Epoch 12/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.1260
Epoch 13/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - loss: 0.1253
Epoch 14/30
6590/6590 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.1249
Epoch 15/30
659

In [18]:
#MSE eval on test set
model.evaluate([X_user_test, X_movie_test], y_test)

#test loss close to training loss, so model is not overfitting
#TEST LOSS - 0.1196

#50-1000 ratings per user
#30 epochs and 1000 training samples
#TRAINING LOSS
#0.15645070374011993 - only genre features
#0.1541982740163803 - genre and decade features
#0.17613600194454193 - movie vector enriched with TMDB features (overfitting, pre dropout)
#0.15593589842319489 - movie vector enriched with TMDB features (overfitting, dropout 0.5)
#0.15448433160781868 - movie vector enriched with TMDB features, decreased dimensionality and more selective threshold (dropout 0.4)
#0.14182472229003906 - additional user features (avg, std, count of ratings)
#0.13969682157039642 - additional TMDB user enriched features (language, country, vote_count, vote_average, revenue)
#0.13761842250823975 - additional user avg SVD features (150)
#0.13769780099391937 - additional user avg overview embeddings therefore ignore 
#0.13835419714450836 - populairty feature added to movie vector

1648/1648 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.1376


0.13758744299411774

In [19]:
y_train_pred_scaled = model.predict([X_user_train, X_movie_train]).flatten()
y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled.reshape(-1, 1)).flatten()

y_test_pred_scaled = model.predict([X_user_test, X_movie_test]).flatten()
y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).flatten()

#target y values
y_train_unscaled = scaler_y.inverse_transform(y_train).flatten()
y_test_unscaled = scaler_y.inverse_transform(y_test).flatten()

#export to xgboost 
np.save("../data/y_train_pred.npy", y_train_pred)
np.save("../data/y_test_pred.npy", y_test_pred)
np.save("../data/y_train_unscaled.npy", y_train_unscaled)
np.save("../data/y_test_unscaled.npy", y_test_unscaled)

6590/6590 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step
1648/1648 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


In [20]:
#ADD RMSE, MAE, R2
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Evaluation function
def evaluate_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return rmse, mae, r2

trainig_rmse, training_mae, training_r2 = evaluate_model(y_train_unscaled, y_train_pred)

print(f"Training RMSE:   {trainig_rmse:.4f}")
print(f"Training MAE:    {training_mae:.4f}")
print(f"Training R²:     {training_r2:.4f}")

print()

test_rmse, test_mae, test_r2 = evaluate_model(y_test_unscaled, y_test_pred)

print(f"Test RMSE:       {test_rmse:.4f}")
print(f"Test MAE:        {test_mae:.4f}")
print(f"Test R²:         {test_r2:.4f}")



Training RMSE:   0.7475
Training MAE:    0.5698
Training R²:     0.4762

Test RMSE:       0.8346
Test MAE:        0.6337
Test R²:         0.3494


In [21]:
#NDCG Evaluation
from sklearn.metrics import ndcg_score

candidate_preds = pd.read_parquet("../data/candidate_preds.parquet")

#relevance scoring
def z_to_relevance(z):
    if z > 1.0:
        return 3  #high link
    elif z > 0.5:
        return 2  #medium link
    elif z > 0.0:
        return 1  #above avergae
    else:
        return 0 #below average

training_ratings = pd.read_csv("../data/train_ratings.csv")
def user_stats(user_id, training_ratings):
    ratings = training_ratings[training_ratings["userId"] == user_id]["rating"]
    return ratings.mean(), ratings.std()

# Create dataframe for relevance scoring of the test set
test_data = pd.read_csv("../data/test_ratings.csv")
sampled_users = training_ratings["userId"].unique()
test_relevance = []
for user_id in sampled_users:
    #test interactions per user
    user_test = test_data[test_data["userId"] == user_id].copy()

    if user_test.empty:
        continue

    # calc user stats from training data
    user_mean, user_std = user_stats(user_id, training_ratings)

    if pd.isna(user_mean) or pd.isna(user_std):
        continue

    # standardise
    user_test["z_score"] = (user_test["rating"] - user_mean) / user_std

    # relevance scoring
    user_test["relevance"] = user_test["z_score"].apply(z_to_relevance)
    test_relevance.append(user_test)

# one dataframe for all users
test_relevance = pd.concat(test_relevance, ignore_index=True)

test_eval = candidate_preds.merge(
    test_relevance[
        ["userId", "movieId", "relevance"]
    ],
    on=["userId", "movieId"],
    how="inner"
)

ncdg_res = []

for user_id, user_df in test_eval.groupby("userId"):

    #need more than 1 item
    if len(user_df) < 2:
        continue

    # convert to np arrays and reshape to 2D for ndcg
    ideal_relevance = user_df["relevance"].values.reshape(1, -1)
    predicted_relevance = user_df["predicted_rating"].values.reshape(1, -1)

    # ndcg score for the user
    ndcg_10 = ndcg_score(ideal_relevance, predicted_relevance, k=10)
    ncdg_res.append({"userId": user_id, "no. user movies": len(user_df), "NDCG@10": ndcg_10})

# back to df
ncdg_res_df = pd.DataFrame(ncdg_res)

print(ncdg_res_df.head(15), "\n")
print(ncdg_res_df.describe())




    userId  no. user movies   NDCG@10
0       94               48  0.690191
1      213               35  0.666887
2      277               26  0.346659
3      457               59  0.612356
4      763               39  0.497666
5     1334               57  0.729898
6     1396               30  0.778484
7     1636               43  0.591053
8     1866               65  0.757993
9     1910               28  0.652210
10    2325               34  0.413779
11    2501               54  0.630014
12    2606               18  0.936379
13    2903               32  0.744856
14    3357              122  0.790595 

              userId  no. user movies      NDCG@10
count    1000.000000      1000.000000  1000.000000
mean   100210.218000        52.718000     0.609755
std     57771.294859        35.964454     0.178286
min        94.000000        10.000000     0.026834
25%     49984.750000        27.000000     0.490989
50%    101870.500000        40.000000     0.615875
75%    148543.500000        67.25

In [22]:
model.save("../models/two_tower.keras")
movieNN.save("../models/movieNN.keras")
userNN.save("../models/userNN.keras")